# YOLOv8m — обучение на 9 классах (Yandex DataSphere)\n\nДатасет: `data/train_all` (~5668 изображений).\n\n**Перед запуском**\n1. Загрузите в проект DataSphere репозиторий (или хотя бы `data/train_all` + `ml/`).\n2. Выберите конфигурацию **с GPU** (например, `g1.1` / V100 / A100).\n3. Запускайте ячейки сверху вниз.

## 1. Установка зависимостей

In [ ]:
%pip install -q ultralytics opencv-python-headless\nimport ultralytics\nprint('ultralytics', ultralytics.__version__)

## 2. Проверка GPU

In [ ]:
from ultralytics import checks\nimport torch\n\nchecks()\nprint('cuda_available:', torch.cuda.is_available())\nif torch.cuda.is_available():\n    print('device:', torch.cuda.get_device_name(0))\nelse:\n    print('WARNING: GPU не найден — обучение будет очень медленным')

## 3. Пути проекта\n\nЕсли ноутбук лежит в `ml/`, корень репозитория — на уровень выше. 

In [ ]:
from pathlib import Path\nimport os\nimport sys\n\n# Корень репозитория (рядом должны быть data/ и ml/)\nCANDIDATES = [\n    Path.cwd(),\n    Path.cwd().parent,\n    Path('/home/jupyter/work/resources'),\n    Path('/home/jupyter'),\n]\n\nROOT = None\nfor p in CANDIDATES:\n    if (p / 'data' / 'train_all' / 'data.yaml').exists():\n        ROOT = p.resolve()\n        break\n\nif ROOT is None:\n    raise FileNotFoundError(\n        'Не найден data/train_all/data.yaml. '\n        'Загрузите датасет в DataSphere и поправьте ROOT.'\n    )\n\nos.chdir(ROOT)\nsys.path.insert(0, str(ROOT))\nprint('ROOT =', ROOT)\nprint('data.yaml =', ROOT / 'data' / 'train_all' / 'data.yaml')

## 4. Запуск обучения\n\nПараметры как в `ml/train_all.py`: yolov8m, 100 эпох, AdamW, cos_lr, save_period=10.

In [ ]:
from ml.train_all import train\n\nOUTPUTS = ROOT / 'outputs'\nOUTPUTS.mkdir(parents=True, exist_ok=True)\n\nbest = train(\n    weights='yolov8m.pt',\n    data=ROOT / 'data' / 'train_all' / 'data.yaml',\n    epochs=100,\n    batch=16,\n    imgsz=640,\n    patience=20,\n    lr0=0.001,\n    warmup_epochs=3,\n    save_period=10,\n    device=None,  # auto: GPU если есть\n    project=ROOT / 'ml' / 'runs' / 'detect',\n    name='train_all_v1',\n    outputs_dir=OUTPUTS,\n)\nprint('DONE best.pt =', best)

## 5. Проверка артефактов\n\n`best.pt` и `last.pt` копируются в `outputs/` для удобной выгрузки из DataSphere.

In [ ]:
from pathlib import Path\n\nfor p in [\n    ROOT / 'outputs' / 'best.pt',\n    ROOT / 'outputs' / 'last.pt',\n    ROOT / 'ml' / 'runs' / 'detect' / 'train_all_v1' / 'weights' / 'best.pt',\n    ROOT / 'ml' / 'runs' / 'detect' / 'train_all_v1' / 'metrics_summary.csv',\n]:\n    print(('OK' if p.exists() else 'MISSING'), p,\n          f'({p.stat().st_size} bytes)' if p.exists() else '')